# 03 — Data Wrangling

**Research question:** now that 02 has decided what should be cleaned, how do you actually clean it — the same way, every time, across three differently-shaped datasets? This stage standardizes D1/D2/D3's artist/song fields (strips collaboration and version-tag noise, unifies casing, unifies column names), then builds a `join_key` so 04 can match records across datasets directly.

**No human judgment needed here.** 02 already made every cleaning decision (`CLEANING_RULES` is its final, hand-authored output) — 03 only executes it. If a cleaning result here looks wrong, the fix belongs back in 02's rules, not in a new judgment call bolted onto 03.

## Architecture

Structurally simpler than 02 — no "machine computes, human judges" split, since judgment already happened upstream:

| | |
|---|---|
| **CONFIG** | `CLEANING_RULES` (copied verbatim from 02's Step 6 output) + `DATASETS` (column names + a `lowercase` flag per dataset) |
| **Engine** | One function, `clean_column()` — reads whichever removal keys exist in a rules entry and applies them, whether it's an artist rule or a song rule |
| **Steps 1-6** | Align column names → apply the rules → report what changed → build `join_key` → check cross-dataset overlap → save |

**Why one function instead of R's six:** R hand-writes a separate cleaning function per dataset per column (`clean_d1_artist`, `clean_d1_song`, `clean_d2_artist`... 6 total), each hardcoded to its own regex list. Here, 02 already normalized every ruleset onto the same schema (`remove_after_marker` / `remove_parens_if_contains` / `remove_version_tags_in_parens` / `remove_version_tags_after_dash` / `remove_brackets_entirely`) — so `clean_column()` just checks which keys are present in the rules dict it's handed, and applies only those steps. One function, six columns.

## Step Blueprint

| Step | What It Does | Question Answered |
|------|------|------|
| **Step 1** | Column Name Alignment | Should D2's `track` and D3's `artist_name`/`track_name` be renamed to match D1's `artist`/`song`? (Yes — 04 needs one shared column name to merge on.) |
| **Step 2** | Apply Cleaning Rules (`clean_column`) | Actually run 02's `CLEANING_RULES` against every column, stripping the collab/version-tag noise that was already decided on. |
| **Step 3** | Cleaning Health Report | How many rows actually changed? What did the change look like? Does the result look reasonable? |
| **Step 4** | Build Join Key | Combine cleaned `artist_clean` + `song_clean` into one key that 04 can match across all three datasets. |
| **Step 5** | Cross-Dataset Key Overlap | Roughly what fraction of D1's keys are findable in D2/D3? A sanity check before 04's real merge. |
| **Step 6** | Save | Persist the cleaned data to `.pkl` so 04 can pick it up. |

**Scope note:** R hand-writes 6 separate cleaning functions (one per dataset per column) — their actual cleaning logic was ported faithfully into `CLEANING_RULES` back in 02, nothing added or removed here. This stage just executes the same rules through one data-driven engine instead of six hardcoded functions.

In [1]:
import pandas as pd
import re

# Load validated datasets from Stage 01
d1_billboard   = pd.read_pickle('../Data/cleaned_D1.pkl')
d2_spotify_all = pd.read_pickle('../Data/cleaned_D2.pkl')
d3_music       = pd.read_pickle('../Data/cleaned_D3.pkl')

print('Datasets loaded')
print(f'D1 (Billboard): {len(d1_billboard):,} rows')
print(f'D2 (Spotify):   {len(d2_spotify_all):,} rows')
print(f'D3 (Music):     {len(d3_music):,} rows')

Datasets loaded
D1 (Billboard): 330,087 rows
D2 (Spotify):   41,106 rows
D3 (Music):     28,372 rows


## CONFIG — `CLEANING_RULES` (from 02's Step 6) + Column Mapping

`CLEANING_RULES` here is copied directly from 02's final Step 6 output, not re-derived — 02's job ends at deciding what should be cleaned, 03 picks up from there and actually cleans it.

The column mapping (`DATASETS`) adds one field 02 didn't need: `lowercase`. **Verified:** R's `clean_d1_artist()`/`clean_d2_artist()` both lowercase, but `clean_d3_artist()` doesn't — initially looked like a possible oversight that could hurt join accuracy. Checked D3's raw `artist_name`/`track_name` directly (28,372 rows): 0 rows contain any uppercase letter — the source data is already fully lowercase, so R's omission was never a real bug. D3 is set to `lowercase: True` here anyway, purely as defensive code in case a future raw CSV update isn't pre-lowercased — zero cost today, since lowercasing already-lowercase text changes nothing.

In [2]:
# ============================================================
# CONFIG -- Only edit this section when adding datasets/rules
# ============================================================

# Copied verbatim from 02_data_pattern_analysis_auto.ipynb Step 6 output (2026/07/20 version).
# Artist-type columns:  remove_after_marker / remove_parens_if_contains / preserve / note
# Song-type columns:    remove_version_tags_in_parens / remove_version_tags_after_dash /
#                        remove_brackets_entirely / preserve / note
CLEANING_RULES = {
    "D1": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with\b"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", ",", "brackets not matching collab markers"],
            "note": "41% of '&' and 52% of ',' cases have a collab marker -- real cleanup needed, not just cosmetic",
        },
        "song": {
            # 2026/07/27 corrected against R's actual clean_d1_song() regex (03_data_wrangling.qmd
            # lines 75-89) -- the previous version here was 02's generalized guess, not a literal
            # match to R's code, and it over-stripped (e.g. bare "live" after a dash, which R never
            # does for D1) and under-stripped ("acoustic"/"unplugged"/"featuring"/"ft." in parens,
            # which R does remove for D1). See references/pipeline-history.md for the full diagnosis.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\."],
            "remove_version_tags_after_dash": [r"remix"],  # R ONLY strips remix after a dash for D1, nothing else
            "remove_brackets_entirely": True,
            "preserve": ["standalone parens/dash treated as title subtitle",
                         "'live'/'part'/'take' as lyric words -- false positive risk, do not blanket-remove"],
            "note": "Corrected 2026/07/27 to match R's clean_d1_song() exactly (see note above)",
        },
    },
    "D2": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with\b"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", "/", "+", "x", "commas"],
            "note": "2026/07/20 evidence: 41% of '&' and 52% of ',' cases have a collab marker, "
                    "59% of parens are collab-related -- essentially the same profile as D1, "
                    "previously missing remove_parens_if_contains (D2 parens were wrongly left as blanket-preserve)",
        },
        "track": {
            # 2026/07/27 corrected against R's actual clean_d2_track() regex (03_data_wrangling.qmd
            # lines 163-182). Two real gaps found: (1) R only removes "(live version|acoustic|unplugged)"
            # from parens, and only removes a dash clause that is EXACTLY "live"/"version live"/"live
            # version" (anchored to end of string) -- the old bare "live" tag here matched ANY parens or
            # dash content containing "live" (e.g. "(Live)", "- Live @ Wacken", "- Live / Take 1"), which
            # R does not touch. (2) R also strips "(featuring|feat.|ft.)" from parens for D2 -- the old
            # config only had "feat\." so "(Featuring X)" was never caught. See references/pipeline-history.md.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\.",
                                               r"radio edit"],
            "remove_version_tags_after_dash": [r"remaster", r"remix", r"radio edit", r"feat\."],
            # bare "live" deliberately dropped -- R only removes it as an EXACT dash-clause match
            # ("- Live" / "- Live Version" and nothing else), which this engine's loose "contains"
            # matching cannot safely replicate without over-matching things like "- Live @ Wacken".
            "remove_brackets_entirely": True,
            "preserve": ["'Part'/'Pt.' after dash -- track numbering, not a version tag"],
            "note": "Corrected 2026/07/27 to match R's clean_d2_track() exactly (see note above)",
        },
    },
    "D3": {
        "artist_name": {
            "remove_after_marker": [],
            "remove_parens_if_contains": [],
            "preserve": ["&", ",", "essentially clean already"],
            "note": "2026/07/20 evidence: 0% Featuring, 100% of '&' cases are Simple (no collab marker), "
                    "only 1 row has parentheses at all (Group Info, preserve) -- no removal rules needed, "
                    "still gets the universal trim/normalize step in 03 like every other column",
        },
        "track_name": {
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"feat\.",
                                               r"live version", r"acoustic", r"unplugged",
                                               r"radio edit", r"album version", r"single version"],
            "remove_version_tags_after_dash": [],
            "remove_brackets_entirely": True,
            "preserve": ["'live' outside parens is frequently a lyric word ('as long as i live') -- "
                          "only the precise phrases above (e.g. 'live version', not bare 'live') get removed"],
            "note": "2026/07/20 evidence: Featuring 23.8%, Performance Version 1.77%, Release Version Tag 0.50% "
                    "all showed up in real classification -- added those two categories that were previously "
                    "missing from the removal list. 0% dash usage, so remove_version_tags_after_dash is empty.",
        },
    },
}

# Column mapping + per-dataset execution settings -- the ONLY thing that
# actually differs between D1 / D2 / D3
DATASETS = {
    "D1": {
        "df": d1_billboard,
        "artist_col": "artist", "song_col": "song",
        "lowercase": True,
        "save_path": r"..\Data\wrangled_D1.pkl",
    },
    "D2": {
        "df": d2_spotify_all,
        "artist_col": "artist", "song_col": "track",
        "lowercase": True,
        "save_path": r"..\Data\wrangled_D2.pkl",
    },
    "D3": {
        "df": d3_music,
        "artist_col": "artist_name", "song_col": "track_name",
        "lowercase": True,  # Verified: D3's raw data has 0 rows with any uppercase letter,
                            # it is already lowercase. Setting True here is defensive code
                            # (zero cost, zero risk), not a fix for a real bug.
        "save_path": r"..\Data\wrangled_D3.pkl",
    },
}

## Engine — No edits needed below this line

`clean_column()` is the only cleaning logic in this stage, and it handles all 6 columns (D1/D2/D3's artist + song). It doesn't know which dataset it's cleaning — only "here's a text column and a rules dict, follow it."

Execution order (mirrors the order inside each of R's `clean_d*_*` functions):
1. Lowercase (if `lowercase` is set) + trim whitespace
2. Remove entire `[...]` brackets (`remove_brackets_entirely`)
3. Remove a whole `(...)` group if its content matches a keyword (`remove_parens_if_contains` / `remove_version_tags_in_parens` — both keys are checked, since artist rules and song rules name this key differently)
4. Strip everything after a bare collaboration marker (`remove_after_marker`)
5. Strip from a `-` dash onward if what follows matches a version tag (`remove_version_tags_after_dash`)
6. Trim whitespace again (every step above can leave stray spaces behind)

`cleaning_health_report()` (used in Step 3) isn't a judgment call — that already happened in 02 — it's a factual before/after check: how many rows changed, and what did the change look like.

In [3]:
def clean_column(series, rules, lowercase=True):
    """Generic column cleaner -- reads ONE CLEANING_RULES entry (from 02's Step 6)
    and applies whichever removal keys are present. Works identically for
    artist-type rules (remove_after_marker / remove_parens_if_contains) and
    song-type rules (remove_version_tags_in_parens / remove_version_tags_after_dash /
    remove_brackets_entirely) -- it just checks which keys exist, does NOT decide
    what should be cleaned (that judgment already happened in 02)."""

    cleaned = series.astype(str)
    if lowercase:
        cleaned = cleaned.str.lower()
    cleaned = cleaned.str.strip()

    # Step a: whole [...] bracket removal
    if rules.get("remove_brackets_entirely"):
        cleaned = cleaned.str.replace(r'\s*\[.*?\]', '', regex=True)

    # Step b: parenthetical content -- remove the WHOLE (...) group if its
    # content matches any of these keywords. Two possible key names because
    # artist rules and song rules ask slightly different questions (02's design).
    paren_tags = rules.get("remove_parens_if_contains", []) + rules.get("remove_version_tags_in_parens", [])
    for tag in paren_tags:
        cleaned = cleaned.str.replace(rf'\s*\([^)]*{tag}[^)]*\)', '', regex=True, case=False)

    # Step c: bare collab marker (not in parentheses) -- strip everything after it
    for marker in rules.get("remove_after_marker", []):
        cleaned = cleaned.str.replace(rf'\s+{marker}.*$', '', regex=True, case=False)

    # Step d: content after a ' - ' dash -- strip from the dash onward if it matches
    # a version tag. Uses the SAME ' - ' (space-hyphen-space) delimiter as 02's
    # detection regex, so what gets removed here matches what Step 4 classified.
    for tag in rules.get("remove_version_tags_after_dash", []):
        cleaned = cleaned.str.replace(rf' - .*{tag}.*$', '', regex=True, case=False)

    cleaned = cleaned.str.strip()
    return cleaned


def cleaning_health_report(df, orig_col, clean_col, n_examples=5):
    """Compare original vs cleaned column -- how much changed, and what did it
    look like. Not a judgment call, just a factual before/after check."""
    changed_mask = df[orig_col] != df[clean_col]
    n_changed = int(changed_mask.sum())
    pct_changed = round(n_changed / len(df) * 100, 2)
    print(f"  Changed: {n_changed:,} rows ({pct_changed}%)")

    examples = df[changed_mask][[orig_col, clean_col]].drop_duplicates().head(n_examples)
    for _, row in examples.iterrows():
        print(f"    '{row[orig_col]}'  ->  '{row[clean_col]}'")

    return n_changed, pct_changed

## Step 1 — Column Name Alignment

D2's `track` becomes `song`, D3's `artist_name`/`track_name` become `artist`/`song` — matching D1 so 04 can merge on one shared column name instead of tracking three different naming schemes.

In [4]:
print("[STEP 1] Column Name Alignment")

RENAME_MAP = {
    "D1": {},                                          # already artist/song
    "D2": {"track": "song"},                            # artist stays artist
    "D3": {"artist_name": "artist", "track_name": "song"},
}

for dataset_name, cfg in DATASETS.items():
    rename_map = RENAME_MAP[dataset_name]
    if rename_map:
        cfg["df"] = cfg["df"].rename(columns=rename_map)
        cfg["artist_col"] = rename_map.get(cfg["artist_col"], cfg["artist_col"])
        cfg["song_col"]   = rename_map.get(cfg["song_col"], cfg["song_col"])
        print(f"  {dataset_name}: renamed {list(rename_map.keys())} -> {list(rename_map.values())}")
    else:
        print(f"  {dataset_name}: no rename needed (already artist/song)")

print("\nColumns now standardized:")
for dataset_name, cfg in DATASETS.items():
    print(f"  {dataset_name}: artist_col='{cfg['artist_col']}', song_col='{cfg['song_col']}'")

[STEP 1] Column Name Alignment
  D1: no rename needed (already artist/song)
  D2: renamed ['track'] -> ['song']
  D3: renamed ['artist_name', 'track_name'] -> ['artist', 'song']

Columns now standardized:
  D1: artist_col='artist', song_col='song'
  D2: artist_col='artist', song_col='song'
  D3: artist_col='artist', song_col='song'


## Step 2 — Apply Cleaning Rules

`CLEANING_RULES`'s keys are still written against the *original* column names (D2's `"track"`, D3's `"artist_name"`/`"track_name"`) — Step 1 already renamed the live columns, so `RULES_KEY_MAP` here maps the current name back to the original CLEANING_RULES key, rather than requiring 02's config itself to change.

In [5]:
print("[STEP 2] Apply Cleaning Rules")

# Maps "current column name" -> "CLEANING_RULES key" per dataset (see Step 1's rename)
RULES_KEY_MAP = {
    "D1": {"artist": "artist", "song": "song"},
    "D2": {"artist": "artist", "song": "track"},
    "D3": {"artist": "artist_name", "song": "track_name"},
}

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]
    lowercase = cfg["lowercase"]
    rules_keys = RULES_KEY_MAP[dataset_name]

    artist_rules = CLEANING_RULES[dataset_name][rules_keys["artist"]]
    song_rules   = CLEANING_RULES[dataset_name][rules_keys["song"]]

    df["artist_clean"] = clean_column(df[artist_col], artist_rules, lowercase=lowercase)
    df["song_clean"]   = clean_column(df[song_col], song_rules, lowercase=lowercase)

    print(f"  {dataset_name}: artist_clean + song_clean created (lowercase={lowercase})")

[STEP 2] Apply Cleaning Rules


  D1: artist_clean + song_clean created (lowercase=True)


  D2: artist_clean + song_clean created (lowercase=True)
  D3: artist_clean + song_clean created (lowercase=True)


## Step 3 — Cleaning Health Report

**Something R never did systematically** — R pastes an ad hoc validation snippet per dataset, eyeballing the first 20 changed rows. Here `cleaning_health_report()` runs the same check uniformly, six times, and doubles as the place to confirm the D3 `lowercase` decision above actually looks reasonable in practice.

In [6]:
print("[STEP 3] Cleaning Health Report")

health_report = {}
for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name}")
    print(f"{'='*55}")

    print(f"\n=== {dataset_name} artist: '{artist_col}' -> 'artist_clean' ===")
    artist_changed, artist_pct = cleaning_health_report(df, artist_col, "artist_clean")

    print(f"\n=== {dataset_name} song: '{song_col}' -> 'song_clean' ===")
    song_changed, song_pct = cleaning_health_report(df, song_col, "song_clean")

    health_report[dataset_name] = {
        "artist_changed": artist_changed, "artist_pct": artist_pct,
        "song_changed": song_changed, "song_pct": song_pct,
    }

print(f"\n{'='*55}")
print("  NOTE on D3 lowercase (2026/07/20 verified)")
print(f"{'='*55}")
print("  R's clean_d3_artist() never lowercased -- looked like a possible oversight vs D1/D2.")
print("  Checked D3's raw artist_name/track_name directly: 0 of 28,372 rows contain any")
print("  uppercase letter at all. The source data is already fully lowercase, so this was")
print("  never a real data-quality bug -- lowercase=True is now set for D3 anyway, purely")
print("  as defensive code in case a future raw CSV update isn't pre-lowercased.")

[STEP 3] Cleaning Health Report

  D1

=== D1 artist: 'artist' -> 'artist_clean' ===
  Changed: 328,951 rows (99.66%)


    'Adele'  ->  'adele'
    'The Kid LAROI & Justin Bieber'  ->  'the kid laroi & justin bieber'
    'Lil Nas X & Jack Harlow'  ->  'lil nas x & jack harlow'
    'Walker Hayes'  ->  'walker hayes'
    'Ed Sheeran'  ->  'ed sheeran'

=== D1 song: 'song' -> 'song_clean' ===
  Changed: 328,971 rows (99.66%)
    'Easy On Me'  ->  'easy on me'
    'Stay'  ->  'stay'
    'Industry Baby'  ->  'industry baby'
    'Fancy Like'  ->  'fancy like'
    'Bad Habits'  ->  'bad habits'

  D2

=== D2 artist: 'artist' -> 'artist_clean' ===
  Changed: 40,937 rows (99.59%)
    'Garland Green'  ->  'garland green'
    'Serge Gainsbourg'  ->  'serge gainsbourg'
    'Lord Melody'  ->  'lord melody'
    'Celia Cruz'  ->  'celia cruz'
    'P. Susheela'  ->  'p. susheela'

=== D2 song: 'song' -> 'song_clean' ===
  Changed: 40,899 rows (99.5%)


    'Jealous Kind Of Fella'  ->  'jealous kind of fella'
    'Initials B.B.'  ->  'initials b.b.'
    'Melody Twist'  ->  'melody twist'
    'Mi Bomba Sonó'  ->  'mi bomba sonó'
    'Uravu Solla'  ->  'uravu solla'

  D3

=== D3 artist: 'artist' -> 'artist_clean' ===
  Changed: 1 rows (0.0%)
    'babes in toyland '  ->  'babes in toyland'

=== D3 song: 'song' -> 'song_clean' ===
  Changed: 408 rows (1.44%)
    'don't look back (feat. van morrison)'  ->  'don't look back'
    'he don't love you [like i love you]'  ->  'he don't love you'
    'you're the song [that i can't stop singing]'  ->  'you're the song'
    'whenever i call you "friend" (feat. stevie nicks)'  ->  'whenever i call you "friend"'
    'i just can't stop loving you (feat. siedah garrett)'  ->  'i just can't stop loving you'

  NOTE on D3 lowercase (2026/07/20 verified)
  R's clean_d3_artist() never lowercased -- looked like a possible oversight vs D1/D2.
  Checked D3's raw artist_name/track_name directly: 0 of 28,372 r

## Step 4 — Build Join Key

`join_key = artist_clean + "|" + song_clean` — `|` as a separator, since it's very unlikely to appear inside an artist or song name itself. 04 can match on this one key instead of comparing artist and song separately.

In [7]:
from IPython.display import display

print("[STEP 4] Join Key Creation")

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    df["join_key"] = df["artist_clean"] + "|" + df["song_clean"]
    print(f"\n{dataset_name} join_key samples:")
    display(df[[cfg["artist_col"], cfg["song_col"], "artist_clean", "song_clean", "join_key"]]
            .drop_duplicates(subset="join_key")
            .head(5))

[STEP 4] Join Key Creation

D1 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,Adele,Easy On Me,adele,easy on me,adele|easy on me
1,The Kid LAROI & Justin Bieber,Stay,the kid laroi & justin bieber,stay,the kid laroi & justin bieber|stay
2,Lil Nas X & Jack Harlow,Industry Baby,lil nas x & jack harlow,industry baby,lil nas x & jack harlow|industry baby
3,Walker Hayes,Fancy Like,walker hayes,fancy like,walker hayes|fancy like
4,Ed Sheeran,Bad Habits,ed sheeran,bad habits,ed sheeran|bad habits



D2 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,Garland Green,Jealous Kind Of Fella,garland green,jealous kind of fella,garland green|jealous kind of fella
1,Serge Gainsbourg,Initials B.B.,serge gainsbourg,initials b.b.,serge gainsbourg|initials b.b.
2,Lord Melody,Melody Twist,lord melody,melody twist,lord melody|melody twist
3,Celia Cruz,Mi Bomba Sonó,celia cruz,mi bomba sonó,celia cruz|mi bomba sonó
4,P. Susheela,Uravu Solla,p. susheela,uravu solla,p. susheela|uravu solla



D3 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,mukesh,mohabbat bhi jhoothi,mukesh,mohabbat bhi jhoothi,mukesh|mohabbat bhi jhoothi
4,frankie laine,i believe,frankie laine,i believe,frankie laine|i believe
6,johnnie ray,cry,johnnie ray,cry,johnnie ray|cry
10,pérez prado,patricia,pérez prado,patricia,pérez prado|patricia
12,giorgos papadopoulos,apopse eida oneiro,giorgos papadopoulos,apopse eida oneiro,giorgos papadopoulos|apopse eida oneiro


## Step 5 — Cross-Dataset Key Overlap

A sanity check before 04's real merge: what fraction of D1's `join_key`s are also findable in D2/D3? R's original version flagged one important number here — **D1-D3's exact match rate is only around 10.6%** — recomputed here to confirm it still holds.

In [8]:
print("[STEP 5] Cross-Dataset Key Overlap")

d1_keys = set(DATASETS["D1"]["df"]["join_key"].unique())
d2_keys = set(DATASETS["D2"]["df"]["join_key"].unique())
d3_keys = set(DATASETS["D3"]["df"]["join_key"].unique())

d1_d2_overlap = len(d1_keys & d2_keys)
d1_d3_overlap = len(d1_keys & d3_keys)

print(f"  D1 unique keys: {len(d1_keys):,}")
print(f"  D2 unique keys: {len(d2_keys):,}")
print(f"  D3 unique keys: {len(d3_keys):,}\n")

print(f"  D1 ∩ D2 overlap: {d1_d2_overlap:,} keys ({round(d1_d2_overlap / len(d1_keys) * 100, 2)}% of D1)")
print(f"  D1 ∩ D3 overlap: {d1_d3_overlap:,} keys ({round(d1_d3_overlap / len(d1_keys) * 100, 2)}% of D1)")

[STEP 5] Cross-Dataset Key Overlap
  D1 unique keys: 29,671
  D2 unique keys: 39,851
  D3 unique keys: 28,342

  D1 ∩ D2 overlap: 20,123 keys (67.82% of D1)
  D1 ∩ D3 overlap: 3,155 keys (10.63% of D1)


## Step 6 — Save

Same pattern as 01's Step 8: saved to `.pkl` for 04 to pick up; no path given means skip, never an accidental overwrite.

In [9]:
print("[STEP 6] Save Wrangled Data")

for dataset_name, cfg in DATASETS.items():
    save_path = cfg.get("save_path")
    if save_path:
        cfg["df"].to_pickle(save_path)
        print(f"  [OK]  {dataset_name} saved to: {save_path}")
    else:
        print(f"  {dataset_name}: no save_path given, skipping.")

[STEP 6] Save Wrangled Data


  [OK]  D1 saved to: ..\Data\wrangled_D1.pkl
  [OK]  D2 saved to: ..\Data\wrangled_D2.pkl
  [OK]  D3 saved to: ..\Data\wrangled_D3.pkl
